In this notebook, we will document and map every found issue with GDPR and AI act. 

# Notebook 03 — Data Governance & Compliance Audit
## NovaCred Credit Application Governance Audit

> **Executive Summary** — This notebook performs a structured compliance audit of the 500 cleaned credit application records produced by Notebook 01. Every data quality issue identified in the cleaning pipeline is mapped to its corresponding obligations under the **General Data Protection Regulation (GDPR)** and the **EU AI Act**, the two primary regulatory frameworks governing the collection, processing, and automated decision-making of personal data in the European Union. For each issue, this notebook documents the legal basis of the violation, assesses its severity, and provides concrete policy recommendations and remediation actions for NovaCred's Data Governance Officer.

### Regulatory Framework

**GDPR (Regulation 2016/679)** governs how personal data of EU individuals is collected, stored, processed, and protected. In the context of credit applications, virtually every field — name, email, SSN, date of birth, income — constitutes personal data and is subject to GDPR obligations including data minimisation, accuracy, storage limitation, and integrity.

**EU AI Act (Regulation 2024/1689)** classifies credit scoring as a **high-risk AI application** under Annex III. This means NovaCred is legally required to ensure the quality of data used to train and operate its credit scoring model, maintain human oversight over automated decisions, and keep detailed documentation of data quality issues and how they were resolved.

### Scope of This Audit

This audit covers the 14 data quality issues catalogued in Notebook 01, organised across five compliance dimensions:

| Dimension | Issues Covered |
|-----------|---------------|
| Uniqueness | Duplicate records, conflicting SSNs |
| Consistency | Gender coding, date formats |
| Validity | Income type, negative credit history, DTI ratio |
| Completeness | Missing income, DOB, email, SSN, timestamp |
| Accuracy | Age plausibility, timestamp plausibility, cross-field plausibility |

> **Note** — This audit uses the cleaned dataset `cleaned_credit_applications.csv` produced by Notebook 01 as its primary input. All flags and audit trail columns created during cleaning are used directly as evidence in this compliance analysis.

In [1]:
import pandas as pd
import regex as re
import datetime
import numpy as np

AUDIT_DATE = datetime.datetime(2026, 3, 1)


df_raw = pd.read_csv('../data/raw_credit_applications.csv')
df_clean = pd.read_csv('../data/cleaned_credit_applications.csv')

## Section 1 — Uniqueness

### #1 Duplicate Records & #2 Conflicting SSNs

### Compliance Analysis — Uniqueness

#### #1 — Duplicate Records

**GDPR — Article 5(1)(d) — Accuracy**
Personal data must be accurate and kept up to date. Duplicate records mean the same individual appears twice in the system, which can lead to incorrect decisions being made on stale or redundant data.

**GDPR — Article 5(1)(c) — Data Minimisation**
Data must be adequate, relevant, and limited to what is necessary. Retaining duplicate records violates this principle — NovaCred should not hold more records about an individual than necessary.

**Recommended Actions:**
- **Data Engineering Team**: Implement a unique constraint on `_id` at the point of data ingestion — reject any application whose ID already exists in the system.
- **Data Engineering Team**: Schedule periodic deduplication audits on the live database to catch any duplicates that slip through.

---

#### #2 — Conflicting SSNs

**Case A — Exact duplicate row (same `_id`, same name)**
This is purely a data pipeline issue, not an SSN conflict. The SSN itself is fine.
- **GDPR — Article 5(1)(c) — Data Minimisation**: redundant records must not be retained.
- **Action**: already resolved by deduplication in Notebook 01. No further action needed on the SSN.

**Case B — Same person, two applications (different `_id`, same name)**
A single person submitted two separate applications. This is not necessarily fraudulent but raises questions about whether both applications were processed independently.
- **GDPR — Article 5(1)(b) — Purpose Limitation**: data collected for one credit application should not be silently reused for a second without the applicant's knowledge.
- **Action**: Both records must be reviewed manually. If the second application was intentional, it should be processed normally. If it was a system error, one record should be removed and the applicant notified.

**Case C — Different people, same SSN (different `_id`, different name)**
Two different individuals claim the same Social Security Number. This is the most serious finding — it is either a data entry error or an indicator of identity fraud.
- **GDPR — Article 5(1)(d) — Accuracy**: at least one of these records contains incorrect personal data.
- **GDPR — Article 33 — Data Breach Notification**: if identity fraud is confirmed, NovaCred is legally obligated to notify the relevant supervisory authority within 72 hours.
- **EU AI Act — Article 10 — Data Governance**: corrupted identity data must not be used in any credit scoring model.
- **Action**: Both records must be immediately excluded from model training and any automated credit decision. The case must be escalated to NovaCred's legal team and, if fraud is confirmed, reported to the relevant authority.

## Section 2 - Consistency

### #3 Gender Coding & #4 Date Formats

### Compliance Analysis — Consistency

#### #3 — Inconsistent Gender Coding

**GDPR — Article 5(1)(d) — Accuracy**
Gender is personal data under GDPR. Storing the same value in four different formats (`Male`, `M`, `Female`, `F`) means the data is not standardised and cannot be reliably processed. Any system that reads this field — including the credit scoring model — may treat `M` and `Male` as different categories, introducing silent errors into automated decisions.

**EU AI Act — Article 10 — Data Quality**
High-risk AI systems must use training data that is consistent and free from encoding errors. Inconsistent gender coding directly violates this requirement — a model trained on this data would receive contradictory signals for what is effectively the same value.

**Consequences if unresolved:**
- The credit scoring model may produce different outcomes for applicants with `M` vs `Male`, constituting an unintended bias.
- Any statistical analysis of gender distribution across approvals/rejections would produce incorrect results, undermining fairness reporting obligations.

**Recommended Actions:**
- **Data Engineering Team**: Implement input validation at ingestion — the `gender` field should only accept `Male` or `Female` (or a defined set of values). Any other value should be rejected at entry.
- **Data Engineering Team**: Add a standardisation step to the ingestion pipeline that maps abbreviations to full words automatically before the record enters the database.

---

#### #4 — Mixed Date Formats

**GDPR — Article 5(1)(d) — Accuracy**
Date of birth is sensitive personal data. When stored in four different formats (`YYYY-MM-DD`, `DD/MM/YYYY`, `MM/DD/YYYY`, `YYYY/MM/DD`), there is a genuine risk of misinterpretation — for example, `03/04/1990` could be read as March 4th or April 3rd depending on the assumed format. This means the system may be operating on an incorrect date of birth for a subset of applicants.

**EU AI Act — Article 10 — Data Quality**
Inconsistent date formats are a data quality defect that directly affects the accuracy of derived features such as age. If the credit scoring model uses age as an input feature, misinterpreted dates of birth will silently corrupt the model's inputs, leading to potentially incorrect credit decisions.

**Consequences if unresolved:**
- Applicants could be incorrectly assessed as older or younger than they are, affecting credit eligibility.
- The 5 records classified as `UNKNOWN` format have dates that could not be reliably interpreted — these applicants' birth dates are effectively unknown to the system despite appearing to have data.

**Recommended Actions:**
- **Data Engineering Team**: Enforce ISO 8601 (`YYYY-MM-DD`) as the only accepted date format at the point of data ingestion. Any other format should be rejected with a clear error message returned to the applicant.
- **Data Engineering Team**: The 5 records with `UNKNOWN` format must be manually reviewed. Their date of birth cannot be reliably used until the correct value is confirmed with the applicant.

# Section 3 - Validity

## #5 Income type, #6 Negative credit history, #7 DTI ratio

### Compliance Analysis — Validity

#### Applicable Regulations

**GDPR — Article 5(1)(d) — Accuracy**
*"Personal data shall be accurate and, where necessary, kept up to date; every reasonable step must be taken to ensure that personal data that are inaccurate, having regard to the purposes for which they are processed, are erased or rectified without delay."*

All three validity issues identified in this section — income stored as text, negative credit history, and an impossible DTI ratio — constitute accuracy violations under this article. In each case, the field contains a value that is either technically incorrect (wrong data type) or factually impossible (negative months, ratio above 1.0), meaning NovaCred is processing and potentially making credit decisions based on inaccurate personal data.

**EU AI Act — Article 10(3) — Data and Data Governance**
*"Training, validation and testing data sets shall be relevant, sufficiently representative, and to the best extent possible, free of errors and complete in view of the intended purpose."*

Credit scoring is classified as a high-risk AI application under Annex III of the EU AI Act. All three issues directly compromise the quality of data used to train and operate the model — a type error in income, an impossible credit history value, and an out-of-range DTI ratio would each corrupt the model's inputs and potentially lead to incorrect or discriminatory credit decisions.

**EU AI Act — Article 14(1) — Human Oversight**
*"High-risk AI systems shall be designed and developed in such a way, including with appropriate human-machine interface tools, that they can be effectively overseen by natural persons during the period in which the AI system is in use."*

Any automated correction applied to these fields — such as the absolute value fix for negative credit history or the division by 10 for the DTI ratio — must be logged, documented, and formally approved by the Governance Officer before the corrected records are used in any model training or credit decision.

---

#### Issue Breakdown

**#5 — Income Stored as Text**
Storing `"55000"` as a string instead of a number means the value cannot be reliably used in numerical computation. A system that fails to convert it correctly would either crash or silently treat the income as null, leading to an incorrect credit assessment. Applicants with text-encoded income risk being assessed as having no income, resulting in a wrongful rejection.

**#6 — Negative Credit History Months**
A value of `-10 months` is factually impossible — credit history cannot be negative. This is either a data entry error or a system bug. If fed directly into the credit scoring model, it would be interpreted as an extremely poor credit profile, leading to an unjust rejection of the affected applicant.

**#7 — Impossible Debt-to-Income Ratio**
A DTI of `1.85` implies the applicant's debt is 185% of their income, which is mathematically impossible as a ratio. The most likely explanation is a decimal point error (`1.85` instead of `0.185`). If uncorrected, the model would treat this applicant as catastrophically over-indebted, virtually guaranteeing an incorrect rejection.

---

#### Consequences if Unresolved
- Applicants affected by these errors could receive incorrect loan rejections based on corrupted data, constituting unfair automated decisions under **GDPR Article 22**.
- The credit scoring model trained on this data would learn distorted relationships between features and creditworthiness, undermining the reliability of all future decisions.
- NovaCred would be in breach of its EU AI Act obligations regarding data quality for high-risk AI systems.

---

#### Recommended Actions
- **Data Engineering Team**: Enforce strict type validation on `annual_income` at ingestion — reject any value that cannot be cleanly cast to a number.
- **Data Engineering Team**: Implement range validation at ingestion — `credit_history_months` must be a non-negative integer and `debt_to_income` must be between `0.0` and `1.0`. Any value outside these ranges should be rejected and flagged for manual review.
- **Governance Officer**: formally approve and document the corrections applied in Notebook 01 (`abs()` for credit history, `÷10` for DTI) before the affected records are used in any model training or credit decision.

# Section 4 - Completeness
## #8 Missing income, DOB, email, SSN, timestamp

### Compliance Analysis — Completeness

#### Applicable Regulations

**GDPR — Article 5(1)(d) — Accuracy**
*"Personal data shall be accurate and, where necessary, kept up to date; every reasonable step must be taken to ensure that personal data that are inaccurate, having regard to the purposes for which they are processed, are erased or rectified without delay."*

Missing personal data fields — date of birth, email, SSN, and processing timestamp — mean that NovaCred is operating with incomplete records. An incomplete record cannot be considered accurate, as the absence of a field is itself a data quality defect that affects the reliability of any decision made on that record.

**GDPR — Article 5(1)(c) — Data Minimisation**
*"Personal data shall be adequate, relevant and limited to what is necessary in relation to the purposes for which they are processed."*

While data minimisation typically concerns collecting too much data, it also applies in reverse — if a field is mandatory for the purpose of credit assessment (such as date of birth or SSN), its absence means the record is inadequate for that purpose and should not be processed further.

**GDPR — Article 22 — Automated Individual Decision-Making**
*"The data subject shall have the right not to be subject to a decision based solely on automated processing, including profiling, which produces legal effects concerning him or her or similarly significantly affects him or her."*

Seven applicants were approved for loans despite having missing email addresses, and four were approved despite missing SSNs. This means NovaCred made legally significant automated credit decisions on records with incomplete identity data — a direct violation of the principle that automated decisions must be based on accurate and complete information.

**EU AI Act — Article 10(3) — Data and Data Governance**
*"Training, validation and testing data sets shall be relevant, sufficiently representative, and to the best extent possible, free of errors and complete in view of the intended purpose."*

Missing values in key fields such as income, date of birth, and SSN directly compromise the completeness requirement for high-risk AI training data. A credit scoring model trained on records with missing identity or financial fields will learn from an incomplete picture of the applicant population, potentially introducing systematic bias into its predictions.

---

#### Issue Breakdown

**#8 — Missing Date of Birth**
Date of birth is PII and cannot be imputed or fabricated. The 4 affected records have been flagged with `dob_missing = True` and their `date_of_birth` and `age` fields preserved as `NaT` and `NaN` respectively. These records must not be used in any age-dependent feature engineering until the correct value is confirmed with the applicant.

**#9 — Missing Email Address**
Email is a primary contact channel for identity verification and regulatory notifications. 7 records have missing email addresses, of which 6 were approved for loans. NovaCred cannot fulfil its GDPR notification obligations (e.g. data breach notification under Article 33, or responding to Subject Access Requests under Article 15) for applicants whose email is unknown.

**#10 — Missing SSN**
SSN is the primary identity verification field in this dataset. 5 records have no SSN, meaning NovaCred cannot uniquely verify the identity of these applicants. Approving a loan without a verified identity is a significant compliance and fraud risk.

**#11 — Missing Processing Timestamp**
400 records have no processing timestamp. While the loan decision itself remains valid, the absence of a timestamp means NovaCred cannot reconstruct when these decisions were made — undermining the audit trail required by the EU AI Act for high-risk automated decisions.

**#12 — Missing Income**
Unlike PII fields, missing income was imputed with the dataset median ($81,000) and flagged with `income_imputed = True`. While this is an acceptable pragmatic fix, imputed values introduce uncertainty into the model's income feature and must be treated with caution in any downstream analysis.

---

#### Consequences if Unresolved
- Applicants with missing PII could receive credit decisions that cannot be legally defended or audited, exposing NovaCred to liability under GDPR Article 22.
- The absence of processing timestamps for 400 records means NovaCred cannot demonstrate compliance with its own decision-making process — a direct breach of EU AI Act transparency requirements.
- A credit scoring model trained on records with missing identity fields may produce biased predictions for applicants with incomplete profiles.

---

#### Recommended Actions
- **Data Engineering Team**: Make `date_of_birth`, `email`, `ssn`, and `processing_timestamp` mandatory fields at ingestion — reject any application that does not provide all four values.
- **Data Engineering Team**: Implement a mandatory timestamp write-back at the point of every loan decision — no decision should be recorded without an associated timestamp.
- **Governance Officer (this report)**: The 4 records with missing DOB, 7 with missing email, and 5 with missing SSN are hereby flagged for mandatory review before use in any model training cycle. The median imputation applied to the 5 missing income records is formally approved, on the condition that `income_imputed = True` is used as a feature flag in all downstream models to signal imputed values.

---

#### A Note on Pseudoanonymisation

Several of the missing PII fields identified in this section — email, SSN, and date of birth — are also candidates for **pseudoanonymisation** under GDPR Article 4(5), which defines pseudoanonymisation as *"the processing of personal data in such a way that the personal data can no longer be attributed to a specific data subject without the use of additional information."*

In the context of this dataset, pseudoanonymisation would involve replacing direct identifiers such as SSN and email with irreversible tokens or hashed values before the data is passed to the data science team for model training. This would ensure that the model never has access to raw PII, reducing NovaCred's exposure in the event of a data breach. A full pseudoanonymisation implementation will be addressed in the next section.

## Privacy protection

*GDPR Article 4(5)*: Pseudonymization means processing personal data so that it can no longer be attributed to a specific data subject without additional information, provided that such additional information is kept separately and is subject to technical and organizational measures

In [2]:
## print(df_clean.columns)
df_clean.iloc[:10, :20].columns

Index(['_id', 'full_name', 'email', 'ssn', 'ip_address', 'gender',
       'date_of_birth', 'age', 'zip_code', 'annual_income', 'income_imputed',
       'credit_history_months', 'credit_history_months_flag', 'debt_to_income',
       'dti_flag', 'savings_balance', 'loan_approved', 'interest_rate',
       'approved_amount', 'rejection_reason'],
      dtype='object')

## Pseudononimization

### Hashing with salt - Full name and email address

In [9]:
import hashlib
import os
import random
import string

# ── Tokenization for full_name ──────────────────────────────
token_lookup = {}  # store this securely, like the salt!

def tokenize(value):
    if value not in token_lookup:
        token = 'TKN_' + ''.join(random.choices(string.digits, k=6))
        token_lookup[value] = token
    return token_lookup[value]

df_clean['full_name'] = df_clean['full_name'].apply(tokenize)

# Save lookup table securely
import json
with open('token_lookup.json', 'w') as f:
    json.dump(token_lookup, f)

# ── Hashing + salt for email ────────────────────────────────
salt = os.urandom(16).hex()

def hash_with_salt(value, salt):
    salted = (salt + str(value)).encode('utf-8')
    return hashlib.sha256(salted).hexdigest()

df_clean['email'] = df_clean['email'].apply(lambda x: hash_with_salt(x, salt))

with open('salt.key', 'w') as f:
    f.write(salt)

In [10]:
df_clean[['_id','full_name','email']].head()

,_id,full_name,email
0,app_200,TKN_903584,4972b3f52a0aeaf108835c565794904f43f4d90ecc831d...
1,app_037,TKN_831561,477e97e14122f50849b7530a62dc2cce57d98ac039b8b9...
2,app_215,TKN_289023,72470eb69f6f6134e6ee235d00d42d0c76e61a1acda773...
3,app_024,TKN_752944,aab6d63a51d32057ada62a7be2eb7189b40b7c9df0620f...
4,app_184,TKN_141570,a9bf09e45ffd7b2531b220c6d85d007de9485556940659...


For **full name**, we applied **tokenization**, replacing each unique name with a randomly 
generated token (e.g., `TKN_482910`), stored in a separate lookup table. This approach was 
chosen because it provides the strongest pseudonymization guarantee, and with it, there is no mathematical 
relationship between the original value and the token, making reverse engineering impossible 
without the lookup table. The key weakness is that the lookup table itself becomes a high-value 
target: if stolen alongside the dataset, re-identification is trivial.

For **email address**, we applied **hashing with salt** (SHA-256). Unlike plain hashing, 
the salt ensures that an attacker cannot pre-compute a dictionary of known email hashes 
to match against our dataset. Email was chosen for this approach rather than tokenization 
because hashing requires no lookup table to maintain, reducing operational risk. The remaining 
weakness is that hashing is deterministic, the same email with the same salt always produces 
the same hash, meaning that if the salt is compromised, all records become vulnerable 
simultaneously.

Both techniques satisfy GDPR pseudonymization requirements under Article 4(5), provided that 
the token lookup table and salt are stored separately from the dataset, access-controlled, 
and not access

### K-anonimization - SSN and Phone number

In [11]:
# ── SSN: suppress first two groups (***-**-6789) ────────────
def suppress_ssn(value):
    parts = str(value).split('-')
    if len(parts) == 3:
        return f"***-**-{parts[2]}"
    return "***-**-****"  # fallback if format is unexpected

df_clean['ssn'] = df_clean['ssn'].apply(suppress_ssn)

# ── IP Address: mask last two octets (192.168.*.*) ──────────
def suppress_ip(value):
    parts = str(value).split('.')
    if len(parts) == 4:
        return f"{parts[0]}.{parts[1]}.*.*"
    return "*.*.*.*"  # fallback

df_clean['ip_address'] = df_clean['ip_address'].apply(suppress_ip)

# ── Processing timestamp: truncate to date only ─────────────
df_clean['processing_timestamp'] = pd.to_datetime(df_clean['processing_timestamp']).dt.date

In [21]:
df_clean[['_id','ssn','ip_address','processing_timestamp']]

,_id,ssn,ip_address,processing_timestamp
0,app_200,***-**-4340,192.168.*.*,2024-01-15T00:00:00Z
1,app_037,***-**-4784,10.1.*.*,NaN
2,app_215,***-**-5178,10.240.*.*,NaN
3,app_024,***-**-1833,192.168.*.*,NaN
4,app_184,***-**-2475,172.29.*.*,2024-01-15T00:00:00Z
...,...,...,...,...
495,app_468,***-**-6099,172.31.*.*,NaN
496,app_192,***-**-8002,172.29.*.*,2024-01-15T00:00:00Z
497,app_234,***-**-8731,10.143.*.*,NaN
498,app_306,***-**-8131,10.26.*.*,NaN


## K-Anonymization Strategy: SSN and IP Address

For **SSN**, we applied **partial suppression**, keeping only the last four digits 
(e.g., `***-**-6789`). This approach is the standard for SSN anonymization, 
familiar from US regulatory practice, and ensures that the most identifying segment - 
the area and group numbers - is removed while preserving just enough structure for 
legitimate record linkage if required. The weakness is that the last four digits alone, 
while insufficient for full re-identification, can still contribute to linkage attacks 
when combined with other quasi-identifiers such as date of birth.

For **IP address**, we masked the last two octets (e.g., `192.168.*.*`), retaining only 
the network prefix. This reduces granularity to the subnet level, making it impossible 
to identify an individual device or household while preserving coarse geographic and 
network-level information useful for fraud detection and audit purposes. The 
risk is that in small or corporate networks, even a subnet prefix may be sufficient to 
narrow down a user to a small group.


We truncated the timestamp to **date precision only** (e.g., 
`2024-03-15 14:32:00` → `2024-03-15`). This preserves temporal context for legitimate 
analytical purposes, such as trend analysis, seasonal patterns in credit applications, 
or audit trails, while eliminating the granularity required for session-level 
re-identification. 

All three transformations achieve **k-anonymization** in the sense that each retained value 
is shared by multiple individuals, preventing unique identification from these fields 
alone. However, true k-anonymization guarantees require verifying that no combination 
of retained fields uniquely identifies any record.

### Quasi-Identifiers - ZIP, gender, birth date

In [17]:
# ── ZIP code: generalize to 3 digits ────────────────────────
df_clean['zip_code'] = df_clean['zip_code'].astype(str).str[:3] + '**'

# ── Age: drop (redundant with date_of_birth) ────────────────
try:
    df_clean.drop(columns=['age'], inplace=True)
except:
    print('Age column already dropped')

# ── Date of birth: keep year only ───────────────────────────
df_clean['date_of_birth'] = pd.to_datetime(df_clean['date_of_birth']).dt.year

# ── Gender: keep as-is (low entropy, acceptable risk) ───────
# no change needed

Age column already dropped


In [18]:
df_clean[['_id','zip_code','date_of_birth']]

,_id,zip_code,date_of_birth
0,app_200,100**,1970.0
1,app_037,100**,1970.0
2,app_215,100**,1970.0
3,app_024,100**,1970.0
4,app_184,100**,1970.0
...,...,...,...
495,app_468,100**,1970.0
496,app_192,100**,1970.0
497,app_234,100**,1970.0
498,app_306,902**,1970.0


Quasi-identifiers are fields that, while not directly identifying on their own, can 
uniquely identify an individual when combined. Following Sweeney (2000), who demonstrated 
that 87% of Americans can be uniquely re-identified using only ZIP code, gender, and date 
of birth, these fields required careful treatment.

For **ZIP code**, we applied **generalization to 3 digits** (e.g., `12345` → `123**`), 
reducing geographic precision to the regional level while preserving enough structure 
for aggregate geographic analysis. Full suppression was rejected as it would eliminate 
useful signal for credit risk modeling across regions.

For **date of birth**, we retained **only the birth year** (e.g., `1990-05-12` → `1990`),
removing month and day which are the primary contributors to uniqueness. This preserves 
age-related analytical value while substantially reducing re-identification risk.

The **age** column was dropped entirely, as it is a derived field calculated directly 
from date of birth. Retaining it alongside a generalized date of birth would be 
contradictory — a precise age value would undermine the generalization already applied 
to the source field.

**Gender** was retained without modification. As a protected attribute under EU 
non-discrimination law and a key variable for AI Act fairness obligations, gender must 
remain available for bias auditing and equitable outcome analysis. Suppressing it would 
not only eliminate critical analytical utility but would also make it impossible to 
detect or correct discriminatory patterns in credit decision outcomes.

### Differential privacy - adding noise to annual income, credit_history_months, debt_to_income, savings_balance, interest_rate, approved_amount

In [14]:
def laplace_interval_str(col, sensitivity, epsilon):
    scale = sensitivity / epsilon
    noise = np.abs(np.random.laplace(0, scale, len(col)))
    return (col - noise).round(0).astype(str) + ' - ' + (col + noise).round(0).astype(str)

epsilon = 1.0

df_clean['annual_income']         = laplace_interval_str(df_clean['annual_income'], 10000, epsilon)
df_clean['savings_balance']       = laplace_interval_str(df_clean['savings_balance'], 5000, epsilon)
df_clean['approved_amount']       = laplace_interval_str(df_clean['approved_amount'], 2000, epsilon)
df_clean['debt_to_income']        = laplace_interval_str(df_clean['debt_to_income'], 0.02, epsilon)
df_clean['interest_rate']         = laplace_interval_str(df_clean['interest_rate'], 0.005, epsilon)
df_clean['credit_history_months'] = laplace_interval_str(df_clean['credit_history_months'], 3, epsilon)

TypeError: unsupported operand type(s) for -: 'str' and 'float'

In [4]:
df_clean[['_id','annual_income', 'age', 'credit_history_months', 'debt_to_income', 'savings_balance', 'interest_rate', 'approved_amount']].head()

,_id,annual_income,age,credit_history_months,debt_to_income,savings_balance,interest_rate,approved_amount
0,app_200,70050.0 - 75949.88,24.0 - 24.36,18.0 - 27.79,0.0 - 0.21,26328.0 - 36096.14,nan - nan,nan - nan
1,app_037,59119.0 - 96881.4,32.0 - 33.86,44.0 - 58.09,0.0 - 0.21,16158.0 - 19672.33,nan - nan,nan - nan
2,app_215,55456.0 - 66544.42,33.0 - 39.01,40.0 - 41.61,0.0 - 0.23,36922.0 - 38895.61,4.0 - 3.71,58555.0 - 59445.19
3,app_024,94715.0 - 111284.69,40.0 - 43.8,64.0 - 75.95,0.0 - 0.38,-2817.0 - 2817.43,4.0 - 4.31,32771.0 - 35228.68
4,app_184,55327.0 - 58673.27,24.0 - 28.02,13.0 - 14.8,0.0 - 0.25,26801.0 - 36725.1,nan - nan,nan - nan


#### Numerical Perturbation via Differential Privacy: Laplace Interval Mechanism

For continuous numerical fields, we applied the **Laplace mechanism** from differential 
privacy theory, replacing each exact value with an interval representing the range of 
plausible true values. Rather than storing a single point estimate, each value 
is expressed as `[value - noise, value + noise]`, where the noise magnitude is drawn 
from a Laplace distribution parameterized by sensitivity and epsilon (ε).

The **sensitivity** parameter reflects the maximum realistic contribution of a single 
individual's record to the field — for example, ±€10,000 for annual income, ±3 months 
for credit history, or ±0.02 for debt-to-income ratio. The **epsilon parameter** was set 
to ε = 1.0, representing a moderate privacy budget that balances re-identification 
protection with analytical utility. Lower values of ε would provide stronger privacy 
guarantees at the cost of wider, less useful intervals.

This approach was applied to: `annual_income`, `savings_balance`, `approved_amount`, 
`debt_to_income`, `interest_rate`, `age`, and `credit_history_months`. Binary and 
categorical fields, as well as flag columns, were excluded as alternation would corrupt 
their logical meaning. The interval representation gives uncertainty
to consumers of the dataset, preventing false precision and discouraging 
exact value reconstruction.

## Final form

In [20]:
df_clean.iloc[:10,:20].head()

,_id,full_name,email,ssn,ip_address,gender,date_of_birth,zip_code,annual_income,income_imputed,credit_history_months,credit_history_months_flag,debt_to_income,dti_flag,savings_balance,loan_approved,interest_rate,approved_amount,rejection_reason,processing_timestamp
0,app_200,TKN_903584,4972b3f52a0aeaf108835c565794904f43f4d90ecc831d...,***-**-4340,192.168.*.*,Male,1970.0,100**,70050.0 - 75949.88,False,18.0 - 27.79,False,0.0 - 0.21,False,26328.0 - 36096.14,False,nan - nan,nan - nan,algorithm_risk_score,2024-01-15T00:00:00Z
1,app_037,TKN_831561,477e97e14122f50849b7530a62dc2cce57d98ac039b8b9...,***-**-4784,10.1.*.*,Male,1970.0,100**,59119.0 - 96881.4,False,44.0 - 58.09,False,0.0 - 0.21,False,16158.0 - 19672.33,False,nan - nan,nan - nan,algorithm_risk_score,NaN
2,app_215,TKN_289023,72470eb69f6f6134e6ee235d00d42d0c76e61a1acda773...,***-**-5178,10.240.*.*,Male,1970.0,100**,55456.0 - 66544.42,False,40.0 - 41.61,False,0.0 - 0.23,False,36922.0 - 38895.61,True,4.0 - 3.71,58555.0 - 59445.19,NaN,NaN
3,app_024,TKN_752944,aab6d63a51d32057ada62a7be2eb7189b40b7c9df0620f...,***-**-1833,192.168.*.*,Male,1970.0,100**,94715.0 - 111284.69,False,64.0 - 75.95,False,0.0 - 0.38,False,-2817.0 - 2817.43,True,4.0 - 4.31,32771.0 - 35228.68,NaN,NaN
4,app_184,TKN_141570,a9bf09e45ffd7b2531b220c6d85d007de9485556940659...,***-**-2475,172.29.*.*,Male,1970.0,100**,55327.0 - 58673.27,False,13.0 - 14.8,False,0.0 - 0.25,False,26801.0 - 36725.1,False,nan - nan,nan - nan,algorithm_risk_score,2024-01-15T00:00:00Z


# Section 5 - Accuracy 
## #Age plausibility, timestamp plausibility, cross-field plausibility

### Compliance Analysis — Accuracy

#### Applicable Regulations

**GDPR — Article 5(1)(d) — Accuracy**
*"Personal data shall be accurate and, where necessary, kept up to date; every reasonable step must be taken to ensure that personal data that are inaccurate, having regard to the purposes for which they are processed, are erased or rectified without delay."*

All three accuracy issues identified in this section involve data that exists and was successfully parsed, but whose values are suspicious or internally contradictory. Unlike completeness issues where data is simply absent, accuracy issues are more dangerous — the data appears valid on the surface but contains hidden errors that could silently corrupt any model or decision built on top of it.

**EU AI Act — Article 10(3) — Data and Data Governance**
*"Training, validation and testing data sets shall be relevant, sufficiently representative, and to the best extent possible, free of errors and complete in view of the intended purpose."*

Credit scoring is a high-risk AI application under Annex III of the EU AI Act. Implausible ages, impossible timestamps, and internally contradictory field combinations are all forms of data error that directly violate the quality requirements for high-risk AI training data. A model trained on these records would learn from corrupted inputs, potentially producing systematically incorrect or discriminatory credit decisions.

**EU AI Act — Article 14(1) — Human Oversight**
*"High-risk AI systems shall be designed and developed in such a way, including with appropriate human-machine interface tools, that they can be effectively overseen by natural persons during the period in which the AI system is in use."*

None of the three issues identified in this section can be safely corrected automatically — each requires human investigation to determine the true value. Flagging without altering is therefore not just a conservative choice but a legal requirement under the AI Act's human oversight obligations.

**EU AI Act — Article 12 — Record-Keeping**
*"High-risk AI systems shall be designed and developed with capabilities enabling the automatic recording of events while the high-risk AI system is in use."*

The 400 records with missing processing timestamps represent a direct breach of this article — NovaCred cannot reconstruct when those credit decisions were made, undermining the audit trail required for high-risk automated decision-making systems.

---

#### Issue Breakdown

**#12 — Age Plausibility**
Applicants younger than 18 cannot legally enter into a credit agreement, and ages above 85 are statistically implausible for this dataset and likely indicate a data entry error such as a wrong century (e.g. `1923` instead of `2023`). Although no violations were found in the current dataset, the check is formally documented here as a standing governance control — any future record with an age outside the valid range `[18, 85]` must be flagged and reviewed before use in any credit decision.

**#13 — Timestamp Plausibility**
Any processing timestamp predating NovaCred's founding year (2020) or postdating the audit date (March 1, 2026) is factually impossible and indicates either a system clock fault or a data entry error. Such timestamps cannot be trusted as evidence of when a decision was made, undermining the audit trail for those records.

**#14 — Cross-Field Plausibility**
Cross-field violations are the most subtle and dangerous accuracy issue in this dataset — they cannot be detected by inspecting any single field in isolation. Three specific contradictions were checked: decision fields inconsistent with the approval outcome, negative savings balances, and rejection reasons contradicted by the applicant's actual DTI ratio. Any of these contradictions, if undetected, would silently corrupt the model's understanding of the relationship between financial features and credit outcomes.

---

#### Consequences if Unresolved
- Applicants with implausible ages could be incorrectly included or excluded from credit eligibility assessments, constituting an unfair automated decision under **GDPR Article 22**.
- Records with impossible timestamps cannot be used as evidence in any regulatory audit or legal challenge, exposing NovaCred to liability under the EU AI Act's record-keeping requirements.
- Cross-field contradictions fed into the credit scoring model would corrupt the learned relationships between features, producing unreliable predictions for the entire applicant population — not just the affected records.

---

#### Recommended Actions
- **Data Engineering Team**: Implement age range validation at ingestion — reject any application where the derived age falls outside `[18, 85]`.
- **Data Engineering Team**: Enforce timestamp validation at ingestion and at every decision write-back — timestamps must fall within the valid operating window and must be recorded automatically at the point of every credit decision.
- **Data Engineering Team**: Implement cross-field validation rules at ingestion — approved applications must always have `interest_rate` and `approved_amount`, rejected applications must always have `rejection_reason`, and `savings_balance` must always be non-negative.
- **Governance Officer (this report)**: The 1 record flagged with `neg_savings_flag = True` (`app_290`, `savings_balance = -5000`) is hereby flagged for mandatory manual review and must be excluded from all model training until the correct value is confirmed. All other cross-field flags with zero violations are formally documented as standing governance controls for future data cycles.